# 10 - YARN

Paste each Bash block into an Ubuntu terminal in order. HDFS and monitoring must be running. This starts ResourceManager, NodeManager and JobHistoryServer and verifies readiness, daemon logs and metrics. The longer MapReduce job and application-log check are optional, separate validation steps.

If returning to this lesson after Spark/Kafka, pause those compute services first with `bash stop-components.sh spark hive kafka`. HDFS, storage and monitoring stay running.


## Open your workspace

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Services: resourcemanager:8088 (web), nodemanager:8042 (web), historyserver:19888 (web). JVM metrics use port 9404 on each Service. NodeManager PVC data-nodemanager-0 is 5 GiB at /data and includes container working directories and logs. ResourceManager/HistoryServer local /data is emptyDir; job history uses HDFS. Native JVM/YARN counters go to Prometheus. NodeManager file-log collection forwards application stdout, stderr and syslog to Loki.

## Check HDFS readiness

```bash
kubectl rollout status statefulset/namenode --timeout=600s
kubectl rollout status statefulset/datanode --timeout=600s
kubectl exec namenode-0 -- hdfs dfsadmin -safemode wait
```



## Apply YARN

```bash
kubectl apply -f monitoring/telemetry-config.yaml
kubectl apply -f hadoop/yarn.yaml
```



## Wait for resourcemanager

```bash
kubectl rollout status statefulset/resourcemanager --timeout=900s
```



## Wait for nodemanager

```bash
kubectl rollout status statefulset/nodemanager --timeout=900s
```



## Wait for historyserver

```bash
kubectl rollout status statefulset/historyserver --timeout=900s
```



## Inspect YARN nodes

```bash
kubectl exec resourcemanager-0 -- yarn node -list
```

Expect one RUNNING NodeManager.

## Check collected logs and metrics

```bash
python3 validations/check-observability.py --components resourcemanager,nodemanager,historyserver --jobs resourcemanager-jmx,nodemanager-jmx,historyserver-jmx
```

The helper makes temporary loopback API forwards, waits for actual log/metric samples and closes its forwards. It fails if collection is missing. Allow several scrape intervals after startup.

## Open the YARN browser

Run this in a second terminal and keep it open. Continue the setup in your original terminal.

```bash
kubectl port-forward --address=127.0.0.1 service/resourcemanager 18088:8088
```

Use http://127.0.0.1:18088 in the Guacamole browser.

## Stop YARN

```bash
bash stop-components.sh yarn
```

HDFS and NodeManager storage remain.

## Resume YARN

```bash
bash start-components.sh yarn
```



## Check resumed nodes

```bash
kubectl exec resourcemanager-0 -- yarn node -list
```



## Pause YARN before the next heavy lesson

```bash
bash stop-components.sh yarn
```

On the 8 GiB profile, pause YARN after these readiness checks. The Hive example uses file loading and simple fetch queries, so it does not require YARN.

## Separate MapReduce validation

The setup is complete above. Run the following optional checks later, in a separate session, while Hive/Spark/Kafka are paused. Resume YARN first with `bash start-components.sh yarn`; pause it again with `bash stop-components.sh yarn` afterward. These application checks are excluded from the normal setup validation. The extraction command is in `validations/README.md`.


## Optional - Run the MapReduce example

```bash
kubectl exec resourcemanager-0 -- bash -ec 'hadoop jar /opt/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar pi 1 1000' > .state/yarn-pi.log 2>&1
cat .state/yarn-pi.log
grep 'Estimated value of Pi' .state/yarn-pi.log
```

This submits an ApplicationMaster and real containers to YARN. It is not a local Python simulation.

## Optional - Inspect completed applications

```bash
kubectl exec resourcemanager-0 -- yarn application -list -appStates ALL
kubectl get pvc data-nodemanager-0
```



## Optional - Check actual YARN application logs

```bash
python3 validations/check-observability.py --components nodemanager --file-logs
```

Requires logs collected from the NodeManager application log directory, not only daemon startup logs.